<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: Chun-Fu Liu
<br>
Github Username: Albert927
<br>
USC ID: 5635413630

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [5]:
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd

Get the AReM Data Set

In [6]:
DATA_DIR = Path(r"C:\Users\liual\Desktop\DSCI 552\HW\hw3 & hw4\AReM")

TIME_SERIES_COLUMNS = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]

RAW_COLUMNS = ["time"] + TIME_SERIES_COLUMNS


def read_arem_csv(path):
    """
    Read and clean one AReM CSV file.

    Cleaning:
    1. Remove comment lines beginning with '#'.
    2. Remove the extra trailing comma in cycling/dataset14.csv.
    3. Detect whether the file is comma- or whitespace-separated.
    4. Restore missing timestamps by linear interpolation.
    """
    
    lines = path.read_text().splitlines()

    data_lines = [
        line.rstrip().rstrip(",")
        for line in lines
        if line.strip() and not line.startswith("#")
    ]

    # bending2/dataset4.csv uses whitespace;
    # the other files use commas.
    separator = "," if "," in data_lines[0] else r"\s+"

    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        header=None,
        names=RAW_COLUMNS,
        sep=separator,
        engine="python",
    )

    # Make sure every value is numeric.
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(how="all")

    # Expected timestamps: 0, 250, ..., 119750.
    # This adds the missing 13500-ms row in sitting/dataset8.csv.
    expected_time = np.arange(0, 120_000, 250)

    df = (
        df.set_index("time")
        .reindex(expected_time)
        .interpolate(method="linear", limit_direction="both")
        .rename_axis("time")
        .reset_index()
    )

    if df.shape != (480, 7):
        raise ValueError(
            f"{path} has unexpected shape {df.shape}"
        )

    if df.isna().any().any():
        raise ValueError(
            f"{path} still contains missing values"
        )

    return df

In [7]:
file_paths = sorted(DATA_DIR.glob("*/dataset*.csv"))

print("Number of CSV files:", len(file_paths))

sample_df = read_arem_csv(file_paths[0])
display(sample_df.head())
print("Sample shape:", sample_df.shape)

Number of CSV files: 88


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


Sample shape: (480, 7)


### (b) Test and Train Data

In [8]:
def belongs_to_test_set(activity, dataset_id):
    if activity in {"bending1", "bending2"}:
        return dataset_id in {1, 2}
    else:
        return dataset_id in {1, 2, 3}


instances = []

for path in file_paths:
    activity = path.parent.name
    dataset_id = int(path.stem.replace("dataset", ""))

    split = (
        "test"
        if belongs_to_test_set(activity, dataset_id)
        else "train"
    )

    instances.append(
        {
            "activity": activity,
            "dataset_id": dataset_id,
            "split": split,
            "path": path,
            "data": read_arem_csv(path),
        }
    )

In [9]:
split_summary = pd.DataFrame(
    [
        {
            "activity": instance["activity"],
            "dataset_id": instance["dataset_id"],
            "split": instance["split"],
        }
        for instance in instances
    ]
)

display(
    split_summary.groupby(["activity", "split"])
    .size()
    .unstack(fill_value=0)
)

print(split_summary["split"].value_counts())

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


split
train    69
test     19
Name: count, dtype: int64


### (c) Feature Extraction

#### i. Research

Commonly used time-domain features for time-series classification
include minimum, maximum, range, mean, median, variance, standard
deviation, first quartile, third quartile, interquartile range,
root mean square, skewness, kurtosis, and zero-crossing rate.

These features summarize different properties of a time series.
For example, the mean and median describe its central tendency,
the standard deviation and interquartile range describe variability,
and the minimum and maximum describe its observed range.

In this assignment, I extract the minimum, maximum, mean, median,
standard deviation, first quartile, and third quartile.

#### ii. Extraction

In [10]:
def extract_time_domain_features(df):
    features = {}

    for series_number, column in enumerate(
        TIME_SERIES_COLUMNS,
        start=1
    ):
        values = df[column]

        features[f"min_{series_number}"] = values.min()
        features[f"max_{series_number}"] = values.max()
        features[f"mean_{series_number}"] = values.mean()
        features[f"median_{series_number}"] = values.median()

        # Sample standard deviation
        features[f"std_{series_number}"] = values.std(ddof=1)

        features[f"1st_quart_{series_number}"] = (
            values.quantile(0.25)
        )
        features[f"3rd_quart_{series_number}"] = (
            values.quantile(0.75)
        )

    return features

In [11]:
feature_rows = []

for instance in instances:
    row = {
        "activity": instance["activity"],
        "dataset_id": instance["dataset_id"],
        "split": instance["split"],
    }

    row.update(
        extract_time_domain_features(instance["data"])
    )

    feature_rows.append(row)


feature_df = (
    pd.DataFrame(feature_rows)
    .sort_values(["activity", "dataset_id"])
    .reset_index(drop=True)
)

metadata_columns = {
    "activity",
    "dataset_id",
    "split",
}

feature_columns = [
    column
    for column in feature_df.columns
    if column not in metadata_columns
]

print("Complete table shape:", feature_df.shape)
print("Number of feature columns:", len(feature_columns))

display(feature_df.head())

Complete table shape: (88, 45)
Number of feature columns: 42


,activity,dataset_id,split,min_1,max_1,mean_1,median_1,std_1,1st_quart_1,3rd_quart_1,...,std_5,1st_quart_5,3rd_quart_5,min_6,max_6,mean_6,median_6,std_6,1st_quart_6,3rd_quart_6
0,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [12]:
rng = np.random.default_rng(552)
n_bootstrap = 10_000

bootstrap_results = []

for feature in feature_columns:
    values = feature_df[feature].to_numpy()

    estimated_std = values.std(ddof=1)

    bootstrap_samples = rng.choice(
        values,
        size=(n_bootstrap, len(values)),
        replace=True,
    )

    bootstrap_std = bootstrap_samples.std(
        axis=1,
        ddof=1,
    )

    ci_lower, ci_upper = np.quantile(
        bootstrap_std,
        [0.05, 0.95],
    )

    bootstrap_results.append(
        {
            "feature": feature,
            "estimated_std": estimated_std,
            "ci_lower_90": ci_lower,
            "ci_upper_90": ci_upper,
        }
    )


bootstrap_df = pd.DataFrame(bootstrap_results)

display(
    bootstrap_df.round(4)
)

,feature,estimated_std,ci_lower_90,ci_upper_90
0,min_1,9.5700,8.2622,10.7614
1,max_1,4.3944,3.3187,5.3026
2,mean_1,5.3357,4.7163,5.8848
3,median_1,5.4401,4.8037,6.0059
4,std_1,1.7722,1.5721,1.9482
5,1st_quart_1,6.1536,5.5747,6.6423
6,3rd_quart_1,5.1389,4.3448,5.8463
7,min_2,0.0000,0.0000,0.0000
8,max_2,5.0627,4.6146,5.4175
9,mean_2,1.5742,1.3911,1.7084


#### iv. Select Features

I select the minimum, mean, and maximum as the three most
important time-domain features.

The mean describes the overall level of a time series, while the
minimum and maximum describe its lower and upper extremes.
Together, these three features provide a simple summary of both
the central tendency and the observed range of each signal.
They are also easy to interpret when comparing different human
activities.

One limitation is that the minimum and maximum can be sensitive
to outliers. Standard deviation or quartiles could be reasonable
alternatives when robustness is more important.

In [13]:
selected_feature_types = ("min_", "mean_", "max_")

selected_columns = [
    column
    for column in feature_columns
    if column.startswith(selected_feature_types)
]

print("Number of selected columns:", len(selected_columns))
print(selected_columns)

Number of selected columns: 18
['min_1', 'max_1', 'mean_1', 'min_2', 'max_2', 'mean_2', 'min_3', 'max_3', 'mean_3', 'min_4', 'max_4', 'mean_4', 'min_5', 'max_5', 'mean_5', 'min_6', 'max_6', 'mean_6']


## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression is expected to have a lower training RSS
than the linear regression.

The cubic model is more flexible and contains the linear model
as a special case when the coefficients of X^2 and X^3 are zero.
Therefore, when both models are fitted by least squares, the
cubic model cannot have a larger training RSS. Its training RSS
will be smaller than or equal to that of the linear model, even
though the true relationship is linear.

### (b) Linear Test

The linear regression is expected to have a lower test RSS.

Because the true relationship is linear, the linear model has
the correct functional form. The cubic model introduces two
unnecessary terms, X^2 and X^3, which may fit random noise in
the training data. This additional flexibility increases
variance without providing a meaningful reduction in bias.

Therefore, although the cubic model can have a lower training
RSS, the linear model is expected to generalize better and have
a lower test RSS.

### (c) Not Linear Train

The cubic regression will have a lower than or equal training
RSS compared with the linear regression.

This conclusion does not depend on whether the true relationship
is linear. The cubic model is more flexible and includes the
linear model as a special case. Consequently, least squares can
always choose cubic coefficients that reproduce the linear fit,
or it can improve the training fit by using the additional terms.

### (d) Not Linear Testing

There is not enough information to determine which model will
have the lower test RSS.

If the true nonlinear relationship is well approximated by a
cubic function, the cubic model may substantially reduce bias
and have a lower test RSS. However, if the departure from
linearity is small, if the true relationship has a different
nonlinear shape, or if the observations contain substantial
noise, the additional flexibility of the cubic model may
increase variance and lead to a higher test RSS.

The result therefore depends on the shape of the true
relationship, the sample size, and the amount of noise.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 